# 03 — Cite evidence and abstain with a policy

**Level:** Beginner · **Estimated time:** 80–100 minutes · **Scenario:** Harborline Support

You will model citations as structured data, compare abstention policies, audit provenance separately from presentation, and test an answerable, ambiguous, and unsupported request.


## How to use this notebook

Work in this order: read the concept, run the deterministic code, change **one** variable, inspect the trace, and write down what changed. The model API is deliberately absent: the learning objective is to understand the evidence system that an LLM would depend on.

**Scenario.** You are building a small, internal assistant for Harborline, a fictional SaaS company. Support needs trustworthy answers about customer communication and production escalation. The corpus is intentionally tiny so every result can be inspected.


## 1. Grounding is a contract, not a citation style

An answer can contain a link and still be unsupported. A trustworthy system preserves evidence identity through retrieval, context construction, generation, validation, and rendering.

```text
question → ranked evidence → policy decision
                    │             ├─ answer: claim + structured citations
                    │             └─ abstain: reason + next safe verification
                    ↓
              provenance audit → presentation
```

The key separation is **data vs. display**. A `Citation` stores a stable chunk ID, source, and retrieval score. Markdown is only a final rendering choice.


In [ ]:
from pathlib import Path
from examples.beginner.citations import AbstentionPolicy, answer_with_citations, audit_answer, render_markdown
from examples.beginner.first_local_rag import load_chunks

ROOT = Path.cwd() if (Path.cwd() / 'examples').exists() else Path('../..')
chunks = load_chunks(ROOT / 'examples/data/beginner-docs')
print('Corpus chunks:', len(chunks))


## 2. A citation does and does not prove things

A citation proves that the system chose a known evidence object. It does **not** independently prove that:

- the passage supports every sentence in the response;
- the source is current;
- the user is authorized to see it;
- the retrieval score is calibrated confidence.

Those properties need separate checks. This notebook implements provenance invariants and an abstention policy; later lessons add permission filters and richer evaluation.


In [ ]:
question = 'What should support do for a confirmed payment incident?'
result = answer_with_citations(question, chunks, top_k=3, min_score=0.20)
print(result)
print('\nRendered for a user:\n')
print(render_markdown(result))


## 3. Audit the evidence object

The audit runs before display logic. It verifies that every cited chunk ID is in the known corpus and reports source diversity and score margin. For a production system, add document version, location, content hash, user authorization, and claim-to-evidence mappings.


In [ ]:
audit = audit_answer(result, chunks)
audit


## 4. Abstention has more than one reason

The assistant can safely decline for different reasons:

| Reason | Meaning | Safe next step |
| --- | --- | --- |
| `insufficient-evidence` | no candidate crossed the evidence threshold | request a source or route to a human |
| `ambiguous-evidence` | top candidates are too close under the policy | retrieve more precisely or clarify the question |
| `insufficient-source-diversity` | a policy requires corroboration but only one source is available | locate a second authoritative source |

The right policy depends on the risk of an unsupported answer. Do not silently convert an abstention into generic model knowledge.


In [ ]:
unsupported = answer_with_citations('What is the capital of France?', chunks)
ambiguous = answer_with_citations(
    'What must an answer distinguish?',
    chunks,
    policy=AbstentionPolicy(min_score=0.10, min_margin=0.90),
)

for label, response in {'unsupported': unsupported, 'ambiguous': ambiguous}.items():
    print(label, '→', response.abstained, response.reason)
    print(render_markdown(response), '\n')


## 5. Thresholds are evaluated, not guessed

Lower thresholds reduce abstentions but may admit weak evidence. Higher thresholds can block useful answers. A margin can flag close competing candidates, but it is still a retrieval heuristic rather than a truth signal.

Test your policy on a balanced set:

- answerable questions with known evidence;
- out-of-corpus questions that must abstain;
- ambiguous questions that need clarification;
- queries using paraphrases, identifiers, and partial wording.


In [ ]:
questions = [
    'How often do enterprise customers receive an update?',
    'Who may restart production services?',
    'What is the capital of France?',
]

for threshold in (0.10, 0.20, 0.50):
    decisions = [answer_with_citations(q, chunks, min_score=threshold) for q in questions]
    print(f'min_score={threshold:.2f}', [(d.abstained, d.reason) for d in decisions])


## 6. Deliberate failure: source existence is not claim support

Ask “Can support restart a service?” The corpus contains `restart`, but the actual rule says support **cannot** restart production services and that approval is required. A system that retrieves a source yet paraphrases it carelessly can cause harm.

The current deterministic baseline returns raw evidence rather than inventing a paraphrase. In a model-backed system, require claim-level evidence, preserve the exact quoted support for high-risk claims, and run factuality evaluation on a held-out set.


In [ ]:
high_risk = answer_with_citations('Can support restart a production service?', chunks)
print(render_markdown(high_risk))
print('\nAudit:', audit_answer(high_risk, chunks))


## 7. Practical policy checklist

- Retrieve only documents the current user is authorized to see.
- Carry stable citation IDs, source locations, versions, and retrieval trace IDs.
- Validate citation IDs against the retrieved evidence set before rendering.
- Make no-answer states explicit, reasoned, and measurable.
- Separate factual claims from recommendations; cite the facts and label the recommendation.
- Evaluate citation correctness, completeness, freshness, and user usefulness—not only whether links render.


## 8. Exercise: write a claim-level response contract

Extend the `CitedAnswer` model with a list of claims. Each claim should contain:

```text
claim text
supporting chunk IDs
claim type: fact | recommendation | unknown
```

Then add tests for:

1. a claim with a citation ID that was never retrieved;
2. an unsupported question that must remain an abstention;
3. a response that separates “the handbook says” from “I recommend”.

**Success criterion:** your validator rejects invalid claim provenance before a user sees the answer.


## 9. Checkpoint and references

1. Why should citations be stored as data before being rendered as Markdown?
2. Name two things a valid citation does not prove.
3. Which policy reason fits a question with no in-corpus evidence?
4. Why should authorization happen before retrieval, not after answer generation?

### References

- [RAG evaluation guide in this repository](../../docs/evaluation.md)
- NIST, [AI Risk Management Framework](https://www.nist.gov/itl/ai-risk-management-framework)
- [RAGAS metrics concepts](https://docs.ragas.io/en/stable/concepts/metrics/)
- [OWASP Top 10 for LLM Applications](https://genai.owasp.org/llm-top-10/)
